# Structural keys

Last time, we dealt with substructures and substructure-based filters. An extension of this concept is assembling a set of several such substructures, which can be searched as a whole against any molecule. As the normal substructure search returns binary results (True or False whether the substructure is contained within the given molecule), searching for a set of substructures within a given molecule produces a set of binary values. This set of binary values can be assembled into a binary vector, where each substructure is represented by a corresponding single bit within the vector. This defined set of substructures, called a structural key or substructure key, allows to characterize any given molecule by a binary vector. Structural keys are basically premade substructure searches that can be used to accelerate structure-related database queries, to quickly compare chemical structures to each other, and to map the feature space of a set of structures, among many other things. More details on structural keys and their applied use are available in the [daylight documentation](https://www.daylight.com/dayhtml/doc/theory/theory.finger.html). Pretty much any computational method that works on binary vectors can be used on structural keys. Let's make a structural key of our own:

In [1]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
import csv

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

In [2]:
# load your own set, and DrugBank :)
with open('../data/chembl_mtor_ic50.csv', 'r') as csvfile:
    reader = csv.DictReader(csvfile, delimiter=";")
    mtor_ligands = [Chem.MolFromSmiles(m['Smiles']) for m in reader]

suppl = Chem.SDMolSupplier('../data/drugbank.sdf')
drugs = [m for m in suppl if m]

# let's assemble a structural key from several substructures that were introduced in the previous exercise 5:

In [3]:
ethanol_pattern = Chem.MolFromSmarts('CCO')
propanol_pattern = Chem.MolFromSmarts('CCCO')
cooh_pattern = Chem.MolFromSmarts('C(=O)[O;h1]')
salicylic_acid_pattern = Chem.MolFromSmarts('c1ccc(c(c1)C(=O)O)O')

So, we have a list of 4 substructures. We can search them all in a molecule, and produce a binary vector of 4 boolean values indicating which substructures are contained in the molecule. Let's characterize the structures in DrugBank using this amazing 4-bit key...

In [4]:
custom_key = [ethanol_pattern, propanol_pattern, cooh_pattern, salicylic_acid_pattern]
mtor_ligands_keys = [[m.HasSubstructMatch(substruct) for substruct in custom_key] for m in mtor_ligands]
len(mtor_ligands_keys), mtor_ligands_keys

(4596,
 [[False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [True, True, False, False],
  [True, False, False, False],
  [False, False, False, False],
  [True, True, False, False],
  [True, True, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [True, True, False, False],
  [True, False, False, False],
  [True, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [True, True, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [False, False, False, False],
  [True, False

As shown above, every molecule in our ligand set is represented by four bits that correspond to the presence of our four custom-defined substructures. This is of course a toy example, and structural keys are often hundreds of bits long. However, you don't have to manually define hundreds of substructures everytime you want to use structural keys. There are several already established sets of substructure features that you can use if you do not have a specific substructures in mind, and want a general characterization of your structure set. You can always extend those generic keys by whatever explicit groups you want to focus on.

# MACCS key
is an often-used structural key consisting of 166 defined [substructures of interest](https://github.com/openbabel/openbabel/blob/master/data/MACCS.txt), also implemented in RDKit. Let's make use of it to characterize our known ligands and the DrugBank database:

In [5]:
from rdkit.Chem import MACCSkeys

In [6]:
mtor_maccs = [MACCSkeys.GenMACCSKeys(m) for m in mtor_ligands]
drugbank_maccs = [MACCSkeys.GenMACCSKeys(m) for m in drugs]
mtor_maccs[0]

As with the custom substructure keys, each molecule is represented by a vector of binary values. In the RDKit implementation, the binary vector is implemented as an RDKit object has some nice [inbuilt binary vector operations](https://www.rdkit.org/docs/source/rdkit.DataStructs.cDataStructs.html):

In [7]:
mtor_maccs[0].GetNumBits() # get size of the vector

167

In [8]:
list(mtor_maccs[0].GetOnBits()) # get indices of the bits that are set to True within the fingerprint

[25,
 32,
 33,
 36,
 42,
 47,
 51,
 55,
 58,
 59,
 60,
 61,
 62,
 64,
 65,
 67,
 69,
 73,
 77,
 80,
 81,
 83,
 87,
 88,
 92,
 94,
 96,
 97,
 98,
 101,
 102,
 105,
 106,
 107,
 110,
 112,
 117,
 120,
 121,
 124,
 125,
 130,
 131,
 133,
 134,
 135,
 136,
 137,
 142,
 144,
 145,
 146,
 148,
 150,
 151,
 154,
 156,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165]

In [9]:
mtor_maccs[0].GetBit(25), mtor_maccs[0].GetBit(26) # getting individual bits

(True, False)

In [10]:
mtor_maccs[0].ToBitString() # write out the bit values in a string

'00000000000000000000000001000000110010000010000100010001001111101101010001000100110100011000101011100110011100101000010011001100001101111100001011101011001010111111110'

... among many other methods. For now, let's use the MACCS substructure keys to compare the relative amounts of substructures within our set of ligands and the DrugBank contents:

In [11]:
mtor_ligands_maccs_sums = [0]*mtor_maccs[0].GetNumBits() # a list of zeros of a given length
for key in mtor_maccs:
    for onbit in key.GetOnBits():
        mtor_ligands_maccs_sums[onbit] += 1
mtor_ligands_maccs_sums

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 68,
 0,
 0,
 115,
 0,
 37,
 0,
 0,
 0,
 907,
 0,
 468,
 0,
 0,
 124,
 216,
 52,
 1604,
 47,
 2,
 11,
 5,
 1,
 0,
 406,
 429,
 12,
 0,
 278,
 1215,
 3884,
 0,
 8,
 119,
 1257,
 910,
 0,
 47,
 26,
 129,
 5,
 24,
 89,
 616,
 974,
 343,
 135,
 625,
 11,
 2528,
 631,
 303,
 631,
 639,
 4451,
 11,
 528,
 4498,
 764,
 639,
 2,
 645,
 56,
 93,
 390,
 633,
 1024,
 3781,
 185,
 4326,
 34,
 2815,
 4293,
 727,
 925,
 2524,
 1809,
 3758,
 3365,
 1404,
 860,
 855,
 681,
 704,
 2203,
 2289,
 1434,
 2626,
 2598,
 1742,
 4532,
 196,
 3490,
 3088,
 733,
 453,
 738,
 3226,
 2851,
 1600,
 1667,
 2395,
 3027,
 3532,
 1479,
 1915,
 1270,
 1557,
 1836,
 3192,
 3473,
 47,
 4565,
 4566,
 4063,
 394,
 1639,
 4520,
 1435,
 2260,
 3292,
 3417,
 859,
 2577,
 2622,
 4290,
 1616,
 4284,
 1314,
 4595,
 3397,
 1012,
 1079,
 1484,
 4518,
 2260,
 3239,
 4569,
 2098,
 3497,
 4189,
 2825,
 3590,
 3916,
 1899,
 3689,
 2774,
 2555,
 4557,
 3435,
 4545,
 3335,
 3911,
 4591,
 4541,
 4595,
 430

In [12]:
drugbank_maccs_sums = [0]*drugbank_maccs[0].GetNumBits() # a list of zeros of a given length
for key in drugbank_maccs:
    for onbit in key.GetOnBits():
        drugbank_maccs_sums[onbit] += 1
drugbank_maccs_sums

[0,
 0,
 0,
 30,
 0,
 1,
 12,
 25,
 99,
 48,
 26,
 117,
 30,
 62,
 33,
 11,
 52,
 77,
 68,
 282,
 7,
 21,
 181,
 212,
 437,
 565,
 227,
 96,
 149,
 727,
 127,
 21,
 431,
 481,
 155,
 44,
 626,
 522,
 1314,
 153,
 167,
 166,
 698,
 1199,
 273,
 249,
 194,
 456,
 873,
 717,
 649,
 601,
 553,
 1898,
 2249,
 663,
 178,
 1636,
 669,
 604,
 685,
 695,
 1756,
 207,
 625,
 2512,
 1118,
 783,
 133,
 1552,
 295,
 528,
 2199,
 755,
 1187,
 2096,
 801,
 2251,
 707,
 2267,
 2303,
 1230,
 2082,
 2787,
 2111,
 2500,
 1686,
 1310,
 1598,
 2481,
 3536,
 3346,
 2829,
 1655,
 1487,
 3195,
 3104,
 2790,
 3061,
 1149,
 2848,
 2779,
 1938,
 786,
 3213,
 2945,
 2672,
 1534,
 1737,
 2378,
 3117,
 3380,
 2732,
 2294,
 1100,
 1831,
 1902,
 3358,
 3353,
 928,
 3229,
 3768,
 2979,
 2796,
 2470,
 3452,
 2556,
 3827,
 2694,
 2854,
 1798,
 4642,
 3561,
 2721,
 1637,
 2200,
 3271,
 4498,
 2585,
 4294,
 3269,
 1357,
 4176,
 3827,
 2807,
 3702,
 4382,
 3402,
 3800,
 2513,
 3945,
 4411,
 3968,
 4557,
 4677,
 4858,
 5261

Since the sizes of our ligand set and the size of the DrugBank database are different, it might be a good idea to divide the raw incidence counts by the total set size, thus getting a number between 0 (never appears in any structure in a set) and 1 (always appears in every structure in a set) for both the ligand and the DrugBank set:

In [13]:
mtor_ligands_maccs_scaled = [x/len(mtor_maccs) for x in mtor_ligands_maccs_sums]
mtor_ligands_maccs_scaled

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.014795474325500435,
 0.0,
 0.0,
 0.02502175805047868,
 0.0,
 0.00805047867711053,
 0.0,
 0.0,
 0.0,
 0.1973455178416014,
 0.0,
 0.10182767624020887,
 0.0,
 0.0,
 0.026979982593559618,
 0.04699738903394256,
 0.011314186248912098,
 0.34899912967798086,
 0.010226283724978242,
 0.0004351610095735422,
 0.002393385552654482,
 0.0010879025239338555,
 0.0002175805047867711,
 0.0,
 0.08833768494342907,
 0.0933420365535248,
 0.0026109660574412533,
 0.0,
 0.060487380330722366,
 0.2643603133159269,
 0.845082680591819,
 0.0,
 0.0017406440382941688,
 0.02589208006962576,
 0.2734986945169713,
 0.19799825935596171,
 0.0,
 0.010226283724978242,
 0.005657093124456049,
 0.028067885117493474,
 0.0010879025239338555,
 0.005221932114882507,
 0.019364664926022627,
 0.134029590948651,
 0.21192341166231507,
 0.07463011314186249,
 0.0293733681462141,
 0.13598781549173194,
 0.002393385552654482,
 0.5500435161009574,
 0.13729329852045258,
 0.06592689295039164,
 0

In [14]:
drugbank_maccs_scaled = [x/len(drugbank_maccs) for x in drugbank_maccs_sums]
drugbank_maccs_scaled

[0.0,
 0.0,
 0.0,
 0.004213483146067416,
 0.0,
 0.0001404494382022472,
 0.0016853932584269663,
 0.0035112359550561797,
 0.013904494382022473,
 0.006741573033707865,
 0.003651685393258427,
 0.01643258426966292,
 0.004213483146067416,
 0.008707865168539325,
 0.0046348314606741575,
 0.001544943820224719,
 0.007303370786516854,
 0.010814606741573033,
 0.009550561797752809,
 0.039606741573033705,
 0.0009831460674157304,
 0.002949438202247191,
 0.025421348314606743,
 0.029775280898876405,
 0.061376404494382024,
 0.07935393258426966,
 0.03188202247191011,
 0.01348314606741573,
 0.020926966292134832,
 0.1021067415730337,
 0.017837078651685392,
 0.002949438202247191,
 0.06053370786516854,
 0.0675561797752809,
 0.021769662921348316,
 0.006179775280898876,
 0.08792134831460674,
 0.07331460674157303,
 0.1845505617977528,
 0.02148876404494382,
 0.02345505617977528,
 0.023314606741573034,
 0.09803370786516855,
 0.16839887640449439,
 0.03834269662921348,
 0.03497191011235955,
 0.027247191011235954,
 

So, we now have the occurrence ratio of each MACCS substructure within our set, and within DrugBank. We can now subtract the two and have a look at the differences:

In [15]:
# compute the differences, store bit numbers prior to sorting
mtor_drugbank_differences = [(i, a_b[0] - a_b[1])
                             for i, a_b in enumerate(zip(mtor_ligands_maccs_scaled, drugbank_maccs_scaled))]
mtor_drugbank_differences

[(0, 0.0),
 (1, 0.0),
 (2, 0.0),
 (3, -0.004213483146067416),
 (4, 0.0),
 (5, -0.0001404494382022472),
 (6, -0.0016853932584269663),
 (7, -0.0035112359550561797),
 (8, 0.0008909799434779625),
 (9, -0.006741573033707865),
 (10, -0.003651685393258427),
 (11, 0.008589173780815758),
 (12, -0.004213483146067416),
 (13, -0.0006573864914287946),
 (14, -0.0046348314606741575),
 (15, -0.001544943820224719),
 (16, -0.007303370786516854),
 (17, 0.18653091110002837),
 (18, -0.009550561797752809),
 (19, 0.06222093466717517),
 (20, -0.0009831460674157304),
 (21, -0.002949438202247191),
 (22, 0.0015586342789528744),
 (23, 0.017222108135066153),
 (24, -0.05006221824546993),
 (25, 0.2696451970937112),
 (26, -0.02165573874693187),
 (27, -0.01304798505784219),
 (28, -0.01853358073948035),
 (29, -0.10101883904909985),
 (30, -0.017619498146898623),
 (31, -0.002949438202247191),
 (32, 0.02780397707826053),
 (33, 0.025785856778243896),
 (34, -0.019158696863907063),
 (35, -0.006179775280898876),
 (36, -0.0274

In [16]:
# let's sort the bits by the difference in MACCS incidence between our ligand set and the DrugBank database
mtor_drugbank_differences.sort(key=lambda x: x[1])
mtor_drugbank_differences

[(139, -0.3828984167962371),
 (90, -0.348456889723355),
 (91, -0.31676714485483226),
 (123, -0.3069699103274953),
 (104, -0.2906896324111832),
 (54, -0.2864974183706398),
 (140, -0.22435984881822005),
 (72, -0.22399191773990085),
 (53, -0.19194292056600268),
 (136, -0.17350932906973332),
 (89, -0.16242372458708598),
 (146, -0.15896553915960138),
 (152, -0.1441179921964385),
 (155, -0.1263851810563167),
 (48, -0.12152445702662794),
 (119, -0.12011079492670715),
 (99, -0.11873062555617486),
 (102, -0.11270450122725184),
 (29, -0.10101883904909985),
 (49, -0.09548031507612872),
 (78, -0.09190001564623855),
 (131, -0.09126133129932235),
 (82, -0.09115376340931536),
 (69, -0.07763810250242026),
 (159, -0.07240272440128692),
 (76, -0.07224760661444735),
 (50, -0.0717870204672358),
 (130, -0.06562643627580408),
 (112, -0.06190629858890484),
 (71, -0.053922316425616804),
 (154, -0.05331370219340703),
 (24, -0.05006221824546993),
 (126, -0.04676073967592731),
 (127, -0.0457680591818973),
 (143,

The MACCS bits that are least prevalent in our MTOR ligand set compared to the DrugBank database contents are 139, 90, 91, 123, 104, 54, etc. These bits correspond to structural patterns [in the MACCS key definition](https://github.com/openbabel/openbabel/blob/master/data/MACCS.txt) "OH", "QHAACH2A" (heteroatom with one H on it, anything, anything, CH2, anything), "QHAAACH2A", "OCO", "QHACH2A" and "QHAAQH", respectively. The most prevalent MACCS bits in our set of MTOR ligands compared to the DrugBank baseline are 62, 38, 65, 77, 135 and 80. These correspond to structural patterns "A\\$!A\\$A" (any atom, cyclic bond, any atom, acyclic bond, any atom, cyclic bond, any atom), "NC(C)N", "C%N" (C, aromatic bond, N), "NAN", "Nnot%A%A" (N, nonaromatic bond, any atom, aromatic bond, any atom) and "NAAAN". Apparently, our MTOR ligand set has, compared to DrugBank, a lot more going on regarding heteroatoms (especially N), aromaticity, and rings in general. How about your set?

# What to do
 - Familiarize yourself with structural keys, if you haven't already in the lectures :) Some resources are Daylight Theory [here](https://www.daylight.com/dayhtml/doc/theory/theory.finger.html) and Dalke's website [here](http://www.dalkescientific.com/writings/NBN/fingerprints.html)
 - Try to make a small structural key of your own, and characterize your set of ligands using this key
 - Have a look at the MACCS key, characterize both your structure set and DrugBank using MACCS key
 - Which MACCS bits are significantly more/less common in your ligand set in comparison to DrugBank?
 - What substructures do these bits [correspond to](https://github.com/openbabel/openbabel/blob/master/data/MACCS.txt)? Some additional materials on how to interpret SMARTS are [here](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html) and [here](https://www.daylight.com/dayhtml_tutorials/languages/smarts/smarts_examples.html).